##### ✅ Problem Statement

You are given an employee table where:

Contact number contains country code + mobile number

You need to:

1. Extract the country code from contact number
1. Mask first 6 digits in contact numbar 
1. Map it to country name using a dictionary (no join, no UDF)
1. Remove country code from contact number
1. Handle inconsistent data formats

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-q-006")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark.sparkContext.setLogLevel("ERROR")

#
# 
#
def display(df):
    df.show(truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 07:32:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://vmware-kubuntu.sandbox.net:4040


In [37]:
#
country_dict = {
    "+91": "India",
    "+01": "United States",
    "+44": "United Kingdom",
    "+61": "Australia"
}

#
data = [
    (1, "+91 9876543210", "john.doe@gmail.com", "Active"),
    (2, "+01 9123456780", "jane.smith@gmail.com", "Inactive"),
    (3, "+44 9988776655", "mike.brown@gmail.com", "Active"),
    (4, "+91 9012345678", "emily.white@gmail.com", "Inactive"),
    (5, "+61 9090909090", "robert.black@gmail.com", "Active")
]


#
columns = ["id", "contact_number", "mail_id", "status"]


#
df = spark.createDataFrame(data, columns)
display(df)



+---+--------------+----------------------+--------+
|id |contact_number|mail_id               |status  |
+---+--------------+----------------------+--------+
|1  |+91 9876543210|john.doe@gmail.com    |Active  |
|2  |+01 9123456780|jane.smith@gmail.com  |Inactive|
|3  |+44 9988776655|mike.brown@gmail.com  |Active  |
|4  |+91 9012345678|emily.white@gmail.com |Inactive|
|5  |+61 9090909090|robert.black@gmail.com|Active  |
+---+--------------+----------------------+--------+



#### Approach - 1 : Using mapPartitions Function

In [43]:
# Using mapPartitions with iterator
def performMasking(partition_data):
    p_data = []
    for record in partition_data:
        #
        contact_number = record.contact_number
        split_number  = contact_number.split(' ')
        country_code  = split_number[0]
        mobile_number = split_number[1]

        unmask_length = 4
        masked_part = '*' * (len(mobile_number) - unmask_length)
        unmasked_part = mobile_number[-unmask_length:]
        masked_number = masked_part + unmasked_part
        full_masked_number = country_code + " " + masked_part + unmasked_part
        # "id", "contact_number", "mail_id", "status"
        p_data.append([record.id, full_masked_number, country_code, masked_number, record.mail_id, record.status])
    return iter(p_data)

# 
masked_columns = ["id", "contact_number", "isd_code", "masked_number", "mail_id", "status"]
masked_df = df.rdd.mapPartitions(performMasking).toDF(masked_columns)
masked_df.show(truncate=False)

+---+--------------+--------+-------------+----------------------+--------+
|id |contact_number|isd_code|masked_number|mail_id               |status  |
+---+--------------+--------+-------------+----------------------+--------+
|1  |+91 ******3210|+91     |******3210   |john.doe@gmail.com    |Active  |
|2  |+01 ******6780|+01     |******6780   |jane.smith@gmail.com  |Inactive|
|3  |+44 ******6655|+44     |******6655   |mike.brown@gmail.com  |Active  |
|4  |+91 ******5678|+91     |******5678   |emily.white@gmail.com |Inactive|
|5  |+61 ******9090|+61     |******9090   |robert.black@gmail.com|Active  |
+---+--------------+--------+-------------+----------------------+--------+



#### Approach - 2 : Using Map Function

In [45]:
# Using mapPartitions with iterator
def performMasking(record):
    #p_data = []
    #for record in partition_data:
        #
        contact_number = record.contact_number
        split_number  = contact_number.split(' ')
        country_code  = split_number[0]
        mobile_number = split_number[1]

        unmask_length = 4
        masked_part = '*' * (len(mobile_number) - unmask_length)
        unmasked_part = mobile_number[-unmask_length:]
        masked_number = masked_part + unmasked_part
        full_masked_number = country_code + " " + masked_part + unmasked_part
        # "id", "contact_number", "mail_id", "status"
        #p_data.append([record.id, full_masked_number, country_code, masked_number, record.mail_id, record.status])
        return (record.id, full_masked_number, country_code, masked_number, record.mail_id, record.status)

# 
masked_columns = ["id", "contact_number", "isd_code", "masked_number", "mail_id", "status"]
masked_df = df.rdd.map(performMasking).toDF(masked_columns)
masked_df.show(truncate=False)

+---+--------------+--------+-------------+----------------------+--------+
|id |contact_number|isd_code|masked_number|mail_id               |status  |
+---+--------------+--------+-------------+----------------------+--------+
|1  |+91 ******3210|+91     |******3210   |john.doe@gmail.com    |Active  |
|2  |+01 ******6780|+01     |******6780   |jane.smith@gmail.com  |Inactive|
|3  |+44 ******6655|+44     |******6655   |mike.brown@gmail.com  |Active  |
|4  |+91 ******5678|+91     |******5678   |emily.white@gmail.com |Inactive|
|5  |+61 ******9090|+61     |******9090   |robert.black@gmail.com|Active  |
+---+--------------+--------+-------------+----------------------+--------+



#### Approach - 3

In [3]:
from pyspark.sql.functions import *
from itertools import chain

country_map = create_map([lit(x) for x in chain(*country_dict.items())])

In [5]:
from pyspark.sql.functions import *

# Split the 'contact_number' column by comma (' ') into an array
split_col = split(df['contact_number'], ' ')


df.withColumn("country_code", split_col.getItem(0)) \
  .withColumn("mobile_number", split_col.getItem(1)) \
  .withColumn("country", country_map[split_col.getItem(0)]) \
  .show()


+---+--------------+--------------------+--------+------------+-------------+--------------+
| id|contact_number|             mail_id|  status|country_code|mobile_number|       country|
+---+--------------+--------------------+--------+------------+-------------+--------------+
|  1|+91 9876543210|  john.doe@gmail.com|  Active|         +91|   9876543210|         India|
|  2|+01 9123456780|jane.smith@gmail.com|Inactive|         +01|   9123456780| United States|
|  3|+44 9988776655|mike.brown@gmail.com|  Active|         +44|   9988776655|United Kingdom|
|  4|+91 9012345678|emily.white@gmail...|Inactive|         +91|   9012345678|         India|
|  5|+61 9090909090|robert.black@gmai...|  Active|         +61|   9090909090|     Australia|
+---+--------------+--------------------+--------+------------+-------------+--------------+



In [44]:
df.select(
    "*",
    (split(col("contact_number")," ")[0]).alias("country_code"),
    (split(col("contact_number")," ")[1]).alias("mobile_number"),
).withColumn("country", country_map[col("country_code")]).show()

+---+--------------+--------------------+--------+------------+-------------+--------------+
| id|contact_number|             mail_id|  status|country_code|mobile_number|       country|
+---+--------------+--------------------+--------+------------+-------------+--------------+
|  1|+91 9876543210|  john.doe@gmail.com|  Active|         +91|   9876543210|         India|
|  2|+01 9123456780|jane.smith@gmail.com|Inactive|         +01|   9123456780| United States|
|  3|+44 9988776655|mike.brown@gmail.com|  Active|         +44|   9988776655|United Kingdom|
|  4|+91 9012345678|emily.white@gmail...|Inactive|         +91|   9012345678|         India|
|  5|+61 9090909090|robert.black@gmai...|  Active|         +61|   9090909090|     Australia|
+---+--------------+--------------------+--------+------------+-------------+--------------+



##### Masking Column

split(df['contact_number'], ' ') =>  ["+91", "9876543210"]

split(df['contact_number'], ' ')[0] => "+91"

split(df['contact_number'], ' ')[1] => "9876543210"

length(split(df['contact_number'], ' ')[1])-4 => 6

repeat(lit("*"), length(split(df['contact_number'], ' ')[1])-4) => ******

right(split(contact_number)[1],lit(4)) => 3210

concat(split(contact_number)[0], lit(" "), repeat(lit("*"), length(split(contact_number)[1])-4), right(split(contact_number)[1],4)) => +91 ******3210


In [ ]:
split_col = split(df['contact_number'], ' ')

# , lit(" "), repeat(lit("*"), 6), right(split_col[1],lit(4))
df.withColumn("masked_numbar",concat(split_col[0], lit(" "), repeat(lit("*"), 6), right(split_col[1],lit(4)))) \
  .show()

+---+--------------+--------------------+--------+--------------+
| id|contact_number|             mail_id|  status| masked_numbar|
+---+--------------+--------------------+--------+--------------+
|  1|+91 9876543210|  john.doe@gmail.com|  Active|+91 ******3210|
|  2|+01 9123456780|jane.smith@gmail.com|Inactive|+01 ******6780|
|  3|+44 9988776655|mike.brown@gmail.com|  Active|+44 ******6655|
|  4|+91 9012345678|emily.white@gmail...|Inactive|+91 ******5678|
|  5|+61 9090909090|robert.black@gmai...|  Active|+61 ******9090|
+---+--------------+--------------------+--------+--------------+



In [36]:
split_col = split(df['contact_number'], ' ')

tlength =  length(split_col[1])-4
print(tlength)

sdf = df.select(length(split_col[1])).collect()[0][0] - 4
print(sdf)
print(type(sdf))

# tlength = 
df.withColumn("length_o", repeat(lit("-"), sdf-2)) \
  .show()

Column<'(length(split(contact_number,  , -1)[1]) - 4)'>
6
<class 'int'>
+---+--------------+--------------------+--------+--------+
| id|contact_number|             mail_id|  status|length_o|
+---+--------------+--------------------+--------+--------+
|  1|+91 9876543210|  john.doe@gmail.com|  Active|    ----|
|  2|+01 9123456780|jane.smith@gmail.com|Inactive|    ----|
|  3|+44 9988776655|mike.brown@gmail.com|  Active|    ----|
|  4|+91 9012345678|emily.white@gmail...|Inactive|    ----|
|  5|+61 9090909090|robert.black@gmai...|  Active|    ----|
+---+--------------+--------------------+--------+--------+



In [27]:
# Get the value of the first row in the column
value = df.select('contact_number').take(1)[0]['contact_number']
print(value)
print(len(value))

+91 9876543210
14


In [24]:
# Get the value of the first row in the column
value = df.select('contact_number').first()['contact_number']
print(value)
print(type(value))

+91 9876543210
<class 'str'>


In [26]:
# Get the value of the first row in the column
value = df.select('contact_number').head(1)[0]['contact_number']
print(value)
print(type(value))

+91 9876543210
<class 'str'>


In [28]:
from pyspark.sql.functions import *

# Split the 'contact_number' column by comma (' ') into an array
split_email = split(df['mail_id'], '\.')
split_lname = split(split_email[1], '@')

df.withColumn("first_name", split_email[0]) \
.withColumn("last_name", split_lname[0]) \
.withColumn("full_name", concat_ws(", ", split_lname[0], split_email[0])) \
.withColumn("full_name_1", concat(split_lname[0], lit(", "), split_email[0])) \
.show()

+---+--------------+--------------------+--------+----------+---------+-------------+-------------+
| id|contact_number|             mail_id|  status|first_name|last_name|    full_name|  full_name_1|
+---+--------------+--------------------+--------+----------+---------+-------------+-------------+
|  1|+91 9876543210|  john.doe@gmail.com|  Active|      john|      doe|    doe, john|    doe, john|
|  2|+01 9123456780|jane.smith@gmail.com|Inactive|      jane|    smith|  smith, jane|  smith, jane|
|  3|+44 9988776655|mike.brown@gmail.com|  Active|      mike|    brown|  brown, mike|  brown, mike|
|  4|+91 9012345678|emily.white@gmail...|Inactive|     emily|    white| white, emily| white, emily|
|  5|+61 9090909090|robert.black@gmai...|  Active|    robert|    black|black, robert|black, robert|
+---+--------------+--------------------+--------+----------+---------+-------------+-------------+



#### Approach - 3 Using UDF

In [ ]:
def mask_sensitive_data(data):
    unmasked_length = 4
    masked_part = '*' * (len(data) - unmasked_length)
    unmasked_part = data[-unmasked_length:]
    return masked_part + unmasked_part

card_number = "1234567890123456"
masked_card = mask_sensitive_data(card_number)
print(masked_card)
# Output: ************3456
